In [ ]:
from pathlib import Path

import pandas as pd
import torch

from learned.stirnet import StirNet
from learned.stirnet.training.checkpoint import load_checkpoint

from learned.stirnet.debugging import (
    DebugConfig,
    StirNetInspector,
    save_debug_trace,
)

from learned.stirnet.debugging.acceptance.first_overfit import (
    _repo_root,
    _reduced_config,
    build_real_batch,
)


REPO_ROOT = _repo_root(Path.cwd())

DATA_DIR = (
    REPO_ROOT
    / "data"
    / "learned"
    / "stirnet"
    / "first_overfit"
    / "BlastoSPIM1_F22_030_034"
)

RUN_DIR = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "first_overfit"
    / "05_same_sample"
)

CHECKPOINT_PATH = RUN_DIR / "checkpoint_step_025.pt"

DEBUG_DIR = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "debug"
    / "step_025_light"
)

DEVICE = torch.device("cuda")

print("Repository:", REPO_ROOT)
print("Data:", DATA_DIR)
print("Checkpoint:", CHECKPOINT_PATH)
print("Debug output:", DEBUG_DIR)
print("CUDA:", torch.cuda.is_available())

assert DATA_DIR.exists()
assert CHECKPOINT_PATH.exists()
assert torch.cuda.is_available()

In [ ]:
batch, sample = build_real_batch(DATA_DIR)

sample

In [ ]:
print("ROI:", sample["roi_shape"])
print("Current cells:", sample["current_count"])
print("GT cells:", sample["target_count"])
print("Graph nodes:", sample["graph_nodes"])
print("Temporal tracklets:", sample["temporal_tracklets"])
print("Required queries:", sample["required_queries"])

In [ ]:
cfg = _reduced_config()

model = StirNet(cfg).to(DEVICE)

checkpoint = load_checkpoint(
    CHECKPOINT_PATH,
    model,
    map_location="cpu",
    strict=True,
)

model.eval()

print("Checkpoint step:", checkpoint.get("step"))
print("Model device:", next(model.parameters()).device)
print("Final exist threshold:", cfg.inference.final_exist_threshold)
print("Mask threshold:", cfg.inference.mask_threshold)

In [ ]:
debug_cfg = DebugConfig.light(
    device="cuda",
    amp_dtype="fp16",
)

inspector = StirNetInspector(
    model,
    debug_cfg,
)

trace = inspector.inspect(batch)

print("Debug forward complete.")
print("Metadata:")
trace.metadata

In [ ]:
save_debug_trace(
    trace,
    DEBUG_DIR,
)

print("Saved to:", DEBUG_DIR)

In [ ]:
query_df = pd.DataFrame(trace.tables["queries"])

print("Rows:", len(query_df))
print("Columns:", len(query_df.columns))

query_df.head()

In [ ]:
query_summary = (
    query_df
    .groupby("query_type")
    .agg(
        total=("query", "count"),
        matched=("matched", "sum"),
        surviving=("survives_final_exist", "sum"),
    )
)

query_summary

In [ ]:
surviving_df = query_df[
    query_df["survives_final_exist"]
].copy()

cols = [
    "query",
    "query_type",
    "source_instance_id",
    "matched",
    "gt_id",
    "layer1_exist_prob",
    "layer2_exist_prob",
    "layer3_exist_prob",
    "center_error_um",
    "layer3_coarse_dice",
]

surviving_df[cols].sort_values(
    "layer3_exist_prob",
    ascending=False,
)

In [ ]:
matched_df = query_df[
    query_df["matched"]
].copy()

center_cols = [
    "query",
    "query_type",
    "gt_id",
    "layer1_center_error_um",
    "layer2_center_error_um",
    "layer3_center_error_um",
    "layer1_exist_prob",
    "layer2_exist_prob",
    "layer3_exist_prob",
]

matched_df[center_cols].sort_values(
    "layer3_center_error_um",
    ascending=False,
).head(15)

In [ ]:
dice_cols = [
    "query",
    "query_type",
    "gt_id",
    "layer1_coarse_dice",
    "layer2_coarse_dice",
    "layer3_coarse_dice",
]

matched_df[dice_cols].sort_values(
    "layer3_coarse_dice",
).head(15)

In [ ]:
deep_cfg = DebugConfig.deep(
    device="cuda",
    amp_dtype="fp16",
    selected_query_indices=(
        73,
        105,
        19,
        41,
        101,
        131,
    ),
    max_selected_queries=6,
)

deep_inspector = StirNetInspector(
    model,
    deep_cfg,
)

deep_trace = deep_inspector.inspect(batch)

print("Deep inspection complete.")
print("Selected queries:", deep_trace.metadata["selected_queries"])

In [ ]:
DEEP_DEBUG_DIR = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "debug"
    / "step_025_deep"
)

save_debug_trace(
    deep_trace,
    DEEP_DEBUG_DIR,
)

print("Saved:", DEEP_DEBUG_DIR)

In [ ]:
mask_df = pd.DataFrame(
    deep_trace.tables["masks"]
)

cols = [
    "query",
    "query_type",
    "gt_id",
    "exist_prob",
    "center_error_um",

    "learned_soft_dice",
    "prior_soft_dice",
    "combined_soft_dice",

    "learned_hard_dice",
    "prior_hard_dice",
    "combined_hard_dice",

    "learned_volume_ratio",
    "prior_volume_ratio",
    "combined_volume_ratio",
]

mask_df[cols]

In [ ]:
import numpy as np

target = batch["targets"][0]

gt_centers_cellscale = (
    torch.as_tensor(
        target["centers_cellscale"]
    )
    .float()
    .cpu()
    .numpy()
)

dref_um = float(batch["dref_um"][0])

def initial_error(row):
    if not row["matched"]:
        return np.nan

    initial_um = np.array(
        [
            row["initial_z_um"],
            row["initial_y_um"],
            row["initial_x_um"],
        ],
        dtype=np.float32,
    )

    target_index = int(row["target_index"])

    gt_um = (
        gt_centers_cellscale[target_index]
        * dref_um
    )

    return float(
        np.linalg.norm(
            initial_um - gt_um
        )
    )

query_df["initial_center_error_um"] = query_df.apply(
    initial_error,
    axis=1,
)

In [ ]:
center_trace_cols = [
    "query",
    "query_type",
    "gt_id",
    "initial_center_error_um",
    "layer1_center_error_um",
    "layer2_center_error_um",
    "layer3_center_error_um",
]

query_df[
    query_df["matched"]
][center_trace_cols].sort_values(
    "layer3_center_error_um",
    ascending=False,
).head(20)

In [ ]:
from learned.stirnet.model.matcher import HungarianMatcher3D
from learned.stirnet.training.trainer import (
    model_forward_from_batch,
    move_to_device,
)

# Rebuild the device batch for one clean eval forward.
b = {}

for key, value in batch.items():
    if key == "targets":
        b[key] = value
    elif key == "spatial_inputs":
        b[key] = value.to(
            device=DEVICE,
            dtype=torch.float16,
        )
    elif key == "instance_labels":
        b[key] = value.to(
            device=DEVICE,
            dtype=torch.int32,
        )
    else:
        b[key] = move_to_device(value, DEVICE)


model.eval()

with torch.no_grad(), torch.autocast(
    device_type="cuda",
    dtype=torch.float16,
):
    outputs = model_forward_from_batch(
        model,
        b,
    )

In [ ]:
matcher = HungarianMatcher3D()

layer_outputs = [
    ("layer1", outputs.aux_outputs[0]),
    ("layer2", outputs.aux_outputs[1]),
    (
        "layer3",
        {
            "exist_logits": outputs.exist_logits,
            "centers_cellscale": outputs.centers_cellscale,
            "coarse_mask_logits": outputs.coarse_mask_logits,
        },
    ),
]

match_maps = {}

for name, out in layer_outputs:
    match = matcher(
        out,
        outputs.query_padding_mask,
        batch["targets"],
    )[0]

    match_maps[name] = {
        int(q): int(gt)
        for q, gt in zip(
            match.pred_indices.detach().cpu(),
            match.target_indices.detach().cpu(),
        )
    }

    print(
        name,
        "matches:",
        len(match_maps[name]),
    )

In [ ]:
m1 = match_maps["layer1"]
m2 = match_maps["layer2"]
m3 = match_maps["layer3"]

final_queries = sorted(m3)

rows = []

gt_ids = (
    torch.as_tensor(
        batch["targets"][0]["ids"]
    )
    .cpu()
    .numpy()
)

query_types = (
    outputs.query_types[0]
    .detach()
    .cpu()
    .numpy()
)

type_names = {
    0: "primary",
    1: "split",
    2: "temporal",
    3: "discovery",
}

for q in final_queries:
    t1 = m1.get(q)
    t2 = m2.get(q)
    t3 = m3.get(q)

    rows.append(
        {
            "query": q,
            "type": type_names[int(query_types[q])],

            "layer1_gt": (
                int(gt_ids[t1])
                if t1 is not None
                else None
            ),
            "layer2_gt": (
                int(gt_ids[t2])
                if t2 is not None
                else None
            ),
            "layer3_gt": (
                int(gt_ids[t3])
                if t3 is not None
                else None
            ),

            "same_1_to_2": (
                t1 is not None
                and t2 is not None
                and t1 == t2
            ),
            "same_2_to_3": (
                t2 is not None
                and t3 is not None
                and t2 == t3
            ),
            "same_all": (
                t1 is not None
                and t2 is not None
                and t3 is not None
                and t1 == t2 == t3
            ),
        }
    )

match_stability_df = pd.DataFrame(rows)

match_stability_df

In [ ]:
print(
    "Final matched queries:",
    len(match_stability_df),
)

print(
    "Same GT layer1 → layer2:",
    match_stability_df["same_1_to_2"].mean(),
)

print(
    "Same GT layer2 → layer3:",
    match_stability_df["same_2_to_3"].mean(),
)

print(
    "Same GT across all layers:",
    match_stability_df["same_all"].mean(),
)

In [ ]:
match_stability_df[
    ~match_stability_df["same_all"]
].sort_values(
    ["type", "query"]
)

In [ ]:
print(
    "\n".join(
        sorted(deep_trace.arrays.keys())
    )
)

In [ ]:
gt_labels = deep_trace.arrays[
    "scene/gt_labels"
]

gt_foreground = gt_labels > 0

foreground_prob = (
    deep_trace.arrays["dense/foreground"]
    .astype(np.float32)
)

foreground_pred = foreground_prob > 0.5

intersection = np.count_nonzero(
    foreground_pred & gt_foreground
)

foreground_dice = (
    2 * intersection
    / (
        np.count_nonzero(foreground_pred)
        + np.count_nonzero(gt_foreground)
        + 1e-8
    )
)

print("CNN foreground Dice:", foreground_dice)
print(
    "Mean probability inside GT:",
    foreground_prob[gt_foreground].mean(),
)
print(
    "Mean probability outside GT:",
    foreground_prob[~gt_foreground].mean(),
)

In [ ]:
boundary_prob = (
    deep_trace.arrays["dense/boundary"]
    .astype(np.float32)
)

gt_boundary = (
    torch.as_tensor(
        batch["targets"][0]["boundary"]
    )
    .cpu()
    .numpy()
    > 0.5
)

boundary_pred = boundary_prob > 0.5

intersection = np.count_nonzero(
    boundary_pred & gt_boundary
)

boundary_dice = (
    2 * intersection
    / (
        np.count_nonzero(boundary_pred)
        + np.count_nonzero(gt_boundary)
        + 1e-8
    )
)

print("CNN boundary Dice:", boundary_dice)
print(
    "Mean probability on GT boundary:",
    boundary_prob[gt_boundary].mean(),
)
print(
    "Mean probability off GT boundary:",
    boundary_prob[~gt_boundary].mean(),
)

In [ ]:
current_labels = (
    batch["instance_labels"][0]
    .cpu()
    .numpy()
)

gt_labels = (
    torch.as_tensor(
        batch["targets"][0]["label_map"]
    )
    .cpu()
    .numpy()
)

source_to_gt = {}

for source_id in np.unique(current_labels):
    if source_id <= 0:
        continue

    source_mask = current_labels == source_id

    overlapping_gt, counts = np.unique(
        gt_labels[source_mask],
        return_counts=True,
    )

    pairs = [
        (int(gt_id), int(count))
        for gt_id, count in zip(
            overlapping_gt,
            counts,
        )
        if gt_id > 0
    ]

    pairs.sort(
        key=lambda x: x[1],
        reverse=True,
    )

    source_to_gt[int(source_id)] = pairs

In [ ]:
seeded_rows = query_df[
    query_df["query_type"].isin(
        ["primary", "split"]
    )
].copy()

rows = []

for _, row in seeded_rows.iterrows():
    source_id = int(
        row["source_instance_id"]
    )

    overlaps = source_to_gt.get(
        source_id,
        [],
    )

    overlapping_ids = [
        gt_id
        for gt_id, _ in overlaps
    ]

    matched_gt = (
        int(row["gt_id"])
        if row["matched"]
        else None
    )

    rows.append(
        {
            "query": int(row["query"]),
            "type": row["query_type"],
            "source_instance": source_id,
            "matched_gt": matched_gt,
            "overlapping_gt": overlapping_ids,
            "overlap_voxels": overlaps,
            "matched_gt_overlaps_source": (
                matched_gt in overlapping_ids
                if matched_gt is not None
                else False
            ),
            "final_exist_prob": float(
                row["layer3_exist_prob"]
            ),
        }
    )

seeded_match_df = pd.DataFrame(rows)

seeded_match_df[
    seeded_match_df["matched"]
    if "matched" in seeded_match_df.columns
    else seeded_match_df["matched_gt"].notna()
]

In [ ]:
seeded_match_df[
    seeded_match_df["matched_gt"].notna()
][
    [
        "query",
        "type",
        "source_instance",
        "matched_gt",
        "overlapping_gt",
        "matched_gt_overlaps_source",
        "final_exist_prob",
    ]
]

In [ ]:
matched_seeded = seeded_match_df[
    seeded_match_df["matched_gt"].notna()
]

print(
    "Matched primary/split:",
    len(matched_seeded),
)

print(
    "Matched to an overlapping GT:",
    matched_seeded[
        "matched_gt_overlaps_source"
    ].sum(),
)

print(
    "Matched to unrelated GT:",
    (
        ~matched_seeded[
            "matched_gt_overlaps_source"
        ]
    ).sum(),
)

In [ ]:
from learned.stirnet import RefinementCriterion
from learned.stirnet.debugging.probes import (
    run_total_backward_probe,
)

criterion = RefinementCriterion(
    cfg.losses,
    cfg.queries,
    cfg.training,
).to(DEVICE)

gradient_probe = run_total_backward_probe(
    model,
    criterion,
    batch,
    device=DEVICE,
    amp_dtype="fp16",
)

In [ ]:
gradient_df = pd.DataFrame(
    gradient_probe["gradients"]
)

gradient_df[
    [
        "module",
        "parameter_norm",
        "gradient_norm",
        "gradient_to_parameter_ratio",
        "gradient_nonzero_count",
        "all_gradients_finite",
    ]
]